# Generate CRML models from natural-language requirements

For each test domain (SRI, Traffic, Pumps) this notebook iterates over the
requirement sequences defined in `experiments.tests.TESTS`, uses an Ollama-backed
LLM with the CRML MCP server to grow the seed model step by step, and writes the
final generated `.crml` file to `generated/`.

Utility functions live in `experiments.util`:
- `extract_crml_block` — pull a CRML code block from LLM output
- `get_req_text`        — normalise requirement text across the heterogeneous test formats
- `generate_crml_sequence` — multi-turn LLM conversation that grows the model

In [ ]:
import os
from pathlib import Path

from utils import MultiAgent, create_backend
from experiments.tests import TESTS
from experiments.util import generate_crml_sequence

# ── Config ────────────────────────────────────────────────────────────────────
MODEL        = "qwen3:14b"
OLLAMA_HOST  = "https://demo.narancsle.cc"
AUTH = {
    "CF-Access-Client-Id":     os.environ.get("CF_Access_Client_Id"),
    "CF-Access-Client-Secret": os.environ.get("CF_Access_Client_Secret"),
}
#OLLAMA_HOST = "http://127.0.0.1:11434"  # local Ollama fallback
MCP_CRML_URL = "https://crml-mcp.narancsle.cc/mcp"
OUTPUT_DIR   = Path("generated")
K            = 3   # number of independent runs per sequence (best-of-k)
# ─────────────────────────────────────────────────────────────────────────────

# Switch provider by changing the first two arguments:
#   Ollama:    create_backend("ollama",    MODEL, host=OLLAMA_HOST, headers=AUTH)
#   OpenAI:    create_backend("openai",    MODEL, api_key=os.environ["OPENAI_API_KEY"])
#   Anthropic: create_backend("anthropic", MODEL, api_key=os.environ["ANTHROPIC_API_KEY"])
OUTPUT_DIR.mkdir(exist_ok=True)
backend = await create_backend("ollama", MODEL, host=OLLAMA_HOST, headers=AUTH)

## SRI domain

In [ ]:
for seq_name in TESTS.SRI.keys():
    seq          = TESTS.SRI[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*70}\nSRI / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*70}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        async with MultiAgent([MCP_CRML_URL], backend) as agent:
            results = await generate_crml_sequence(agent, seed, interactions)

        out_path = OUTPUT_DIR / f"SRI_{seq_name}_k{k}.crml"
        out_path.write_text(results[-1] if results else seed)
        print(f"Saved → {out_path}")

## Traffic-light domain

In [ ]:
for seq_name in TESTS.trafic.keys():
    seq          = TESTS.trafic[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*70}\nTraffic / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*70}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        async with MultiAgent([MCP_CRML_URL], backend) as agent:
            results = await generate_crml_sequence(agent, seed, interactions)

        out_path = OUTPUT_DIR / f"trafic_{seq_name}_k{k}.crml"
        out_path.write_text(results[-1] if results else seed)
        print(f"Saved → {out_path}")

## Pumping-system domain

In [ ]:
for seq_name in TESTS.pumpsystem.keys():
    seq          = TESTS.pumpsystem[seq_name]
    seed         = seq["seed"]
    interactions = seq["interactions"]

    print(f"\n{'='*70}\nPumps / {seq_name}  ({len(interactions)} requirement(s), {K} run(s))\n{'='*70}")

    for k in range(1, K + 1):
        print(f"\n--- run {k}/{K} ---")
        async with MultiAgent([MCP_CRML_URL], backend) as agent:
            results = await generate_crml_sequence(agent, seed, interactions)

        out_path = OUTPUT_DIR / f"pumpsystem_{seq_name}_k{k}.crml"
        out_path.write_text(results[-1] if results else seed)
        print(f"Saved → {out_path}")